In [1]:
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.llms import OpenAI
from langchain import hub
from langchain.tools.render import render_text_description
from dotenv import load_dotenv
from tools.ffmpeg_tool import FfmpegTool

load_dotenv()

/Users/johannes/Repos/Tapped/music_analysis_agent/.venv/lib/python3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


True

In [2]:
llm = OpenAI(temperature=0)
tools = load_tools(["serpapi", "llm-math"], llm=llm)
tools.append(FfmpegTool())
# tools.append(MusicInformationRetrievalTool())
# tools.append(ReverbTool)

In [3]:
prompt = hub.pull("hwchase17/react")
prompt = prompt.partial(
    tools=render_text_description(tools),
    tool_names=", ".join([t.name for t in tools]),
)

In [4]:
agent_executor = initialize_agent(
    tools, 
    llm, 
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, 
    verbose=True,
    handle_parsing_errors=True, 
)

In [6]:
agent_executor.invoke(
    {
        "input": "can you clip the audio file located at input.wav to be 5 seconds, reverse it, then save it as an mp3 file?",
    }
)



> Entering new AgentExecutor chain...
 I need to edit and manipulate the audio file
Action: FfmpegTool
Action Input: -i input.wav -t 5 -af areverse output.mp3

running command: ffmpeg -loglevel error -y -i input.wav -t 5 -af areverse output.mp3
[ffmpeg exited with 0]

Observation: 
Thought: I now know the final answer
Final Answer: Yes, you can clip the audio file located at input.wav to be 5 seconds, reverse it, then save it as an mp3 file.

> Finished chain.


{'input': 'can you clip the audio file located at input.wav to be 5 seconds, reverse it, then save it as an mp3 file?',
 'output': 'Yes, you can clip the audio file located at input.wav to be 5 seconds, reverse it, then save it as an mp3 file.'}